# Phase 7 — Final Model Packaging & Prediction Testing

This notebook prepares the selected employee attrition model for API deployment.

## Objectives

- Load the cleaned dataset
- Load `best_model.joblib`
- Verify the model pipeline
- Test predictions using raw employee records
- Inspect prediction probabilities
- Create an API input schema
- Save model metadata
- Save sample API request payloads
- Verify that the final model can be loaded independently

In [1]:
from pathlib import Path
import json
import platform
import warnings

import joblib
import numpy as np
import pandas as pd
import sklearn

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

TARGET_COLUMN = "Attrition"

print("Libraries imported successfully.")
print("Python:", platform.python_version())
print("scikit-learn:", sklearn.__version__)

Libraries imported successfully.
Python: 3.14.4
scikit-learn: 1.9.0


## 1. Configure Project Paths

In [2]:
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "clean_employee_attrition.csv"
)

MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
API_DIR = PROJECT_ROOT / "api"

BEST_MODEL_PATH = MODELS_DIR / "best_model.joblib"
BEST_MODEL_INFO_PATH = REPORTS_DIR / "best_model_info.json"

MODEL_METADATA_PATH = MODELS_DIR / "model_metadata.json"
INPUT_SCHEMA_PATH = MODELS_DIR / "input_schema.json"
SAMPLE_REQUEST_PATH = API_DIR / "sample_request.json"

MODELS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
API_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT.resolve())
print("Dataset path:", DATA_PATH.resolve())
print("Best model path:", BEST_MODEL_PATH.resolve())

Project root: C:\Users\Nasteho Abdi\employee-attrition-prediction
Dataset path: C:\Users\Nasteho Abdi\employee-attrition-prediction\data\processed\clean_employee_attrition.csv
Best model path: C:\Users\Nasteho Abdi\employee-attrition-prediction\models\best_model.joblib


## 2. Validate Required Files

In [3]:
required_files = {
    "Cleaned dataset": DATA_PATH,
    "Best model": BEST_MODEL_PATH,
}

missing_files = [
    f"{name}: {path.resolve()}"
    for name, path in required_files.items()
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Required files were not found:\n\n"
        + "\n".join(missing_files)
        + "\n\nRun the previous phases first."
    )

print("All required files were found.")

All required files were found.


## 3. Load the Dataset and Final Model

In [4]:
df = pd.read_csv(DATA_PATH)
best_model = joblib.load(BEST_MODEL_PATH)

print("Dataset shape:", df.shape)
print("Loaded model type:", type(best_model))

Dataset shape: (1470, 31)
Loaded model type: <class 'sklearn.pipeline.Pipeline'>


In [5]:
if TARGET_COLUMN not in df.columns:
    raise ValueError(
        f"Target column '{TARGET_COLUMN}' was not found."
    )

X = df.drop(columns=[TARGET_COLUMN]).copy()
y = df[TARGET_COLUMN].copy()

print("Input feature count:", X.shape[1])
print("Target values:", sorted(y.unique().tolist()))

Input feature count: 30
Target values: [0, 1]


## 4. Inspect the Final Pipeline

In [6]:
if hasattr(best_model, "named_steps"):
    print("Pipeline steps:")
    for step_name, step_object in best_model.named_steps.items():
        print(f"- {step_name}: {type(step_object).__name__}")
else:
    print(
        "The saved object is not a scikit-learn Pipeline. "
        "It may still work, but a complete preprocessing + model pipeline is preferred."
    )

Pipeline steps:
- preprocessor: ColumnTransformer
- model: LogisticRegression


## 5. Verify Batch Predictions

In [7]:
sample_batch = X.head(10).copy()

batch_predictions = best_model.predict(sample_batch)

if hasattr(best_model, "predict_proba"):
    batch_probabilities = best_model.predict_proba(sample_batch)[:, 1]
else:
    batch_probabilities = np.full(
        len(sample_batch),
        np.nan,
    )

prediction_preview = sample_batch.copy()
prediction_preview["ActualAttrition"] = y.head(10).values
prediction_preview["PredictedAttrition"] = batch_predictions
prediction_preview["AttritionProbability"] = np.round(
    batch_probabilities,
    4,
)

prediction_preview[
    [
        "ActualAttrition",
        "PredictedAttrition",
        "AttritionProbability",
    ]
]

,ActualAttrition,PredictedAttrition,AttritionProbability
0,1,1,0.7244
1,0,0,0.0147
2,1,0,0.4839
3,0,0,0.1041
4,0,0,0.4134
5,0,0,0.0808
6,0,0,0.0868
7,0,0,0.1212
8,0,0,0.0792
9,0,0,0.0319


## 6. Verify a Single Raw Employee Prediction

The API will receive one employee record at a time. The input is converted into a one-row pandas DataFrame before prediction.

In [8]:
single_employee = X.iloc[0].to_dict()

single_employee_df = pd.DataFrame(
    [single_employee]
)

single_prediction = int(
    best_model.predict(single_employee_df)[0]
)

single_probability = (
    float(
        best_model.predict_proba(
            single_employee_df
        )[0, 1]
    )
    if hasattr(best_model, "predict_proba")
    else None
)

single_result = {
    "prediction": single_prediction,
    "prediction_label": (
        "Likely to Leave"
        if single_prediction == 1
        else "Likely to Stay"
    ),
    "attrition_probability": single_probability,
}

single_result

{'prediction': 1,
 'prediction_label': 'Likely to Leave',
 'attrition_probability': 0.7243999205399878}

## 7. Create a Human-Readable Risk Level

In [9]:
def probability_to_risk_level(
    probability: float | None,
) -> str:
    if probability is None:
        return "Unknown"

    if probability < 0.30:
        return "Low"
    elif probability < 0.60:
        return "Medium"
    else:
        return "High"


single_result["risk_level"] = (
    probability_to_risk_level(
        single_probability
    )
)

single_result

{'prediction': 1,
 'prediction_label': 'Likely to Leave',
 'attrition_probability': 0.7243999205399878,
 'risk_level': 'High'}

## 8. Inspect Input Columns and Data Types

In [10]:
input_feature_report = pd.DataFrame({
    "feature": X.columns,
    "pandas_dtype": X.dtypes.astype(str).values,
    "sample_value": [
        X[column].dropna().iloc[0]
        if not X[column].dropna().empty
        else None
        for column in X.columns
    ],
    "required": True,
})

input_feature_report

,feature,pandas_dtype,sample_value,required
0,Age,int64,41,True
1,BusinessTravel,str,Travel_Rarely,True
2,DailyRate,int64,1102,True
3,Department,str,Sales,True
4,DistanceFromHome,int64,1,True
5,Education,int64,2,True
6,EducationField,str,Life Sciences,True
7,EnvironmentSatisfaction,int64,2,True
8,Gender,str,Female,True
9,HourlyRate,int64,94,True


## 9. Generate API Input Schema

The schema records:

- Feature name
- Data type
- Whether the field is required
- Example value
- Valid categorical values where available

In [11]:
def infer_json_type(dtype) -> str:
    if pd.api.types.is_integer_dtype(dtype):
        return "integer"
    if pd.api.types.is_float_dtype(dtype):
        return "number"
    if pd.api.types.is_bool_dtype(dtype):
        return "boolean"
    return "string"


input_schema = {
    "title": "Employee Attrition Prediction Input",
    "target_column": TARGET_COLUMN,
    "feature_count": int(X.shape[1]),
    "features": [],
}

for column in X.columns:
    feature_info = {
        "name": column,
        "type": infer_json_type(X[column].dtype),
        "required": True,
    }

    non_null_values = X[column].dropna()

    if not non_null_values.empty:
        example_value = non_null_values.iloc[0]

        if isinstance(example_value, np.integer):
            example_value = int(example_value)
        elif isinstance(example_value, np.floating):
            example_value = float(example_value)
        elif isinstance(example_value, np.bool_):
            example_value = bool(example_value)
        else:
            example_value = str(example_value)

        feature_info["example"] = example_value

    if (
        feature_info["type"] == "string"
        and X[column].nunique(dropna=True) <= 30
    ):
        feature_info["allowed_values"] = sorted(
            X[column]
            .dropna()
            .astype(str)
            .unique()
            .tolist()
        )

    input_schema["features"].append(feature_info)

with INPUT_SCHEMA_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        input_schema,
        file,
        indent=4,
    )

print(
    "Input schema saved to:",
    INPUT_SCHEMA_PATH.resolve(),
)

Input schema saved to: C:\Users\Nasteho Abdi\employee-attrition-prediction\models\input_schema.json


## 10. Generate a Sample FastAPI Request

In [12]:
sample_request = {}

for column in X.columns:
    value = X[column].dropna().iloc[0]

    if isinstance(value, np.integer):
        value = int(value)
    elif isinstance(value, np.floating):
        value = float(value)
    elif isinstance(value, np.bool_):
        value = bool(value)
    else:
        value = str(value)

    sample_request[column] = value

with SAMPLE_REQUEST_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        sample_request,
        file,
        indent=4,
    )

print(
    "Sample API request saved to:",
    SAMPLE_REQUEST_PATH.resolve(),
)

sample_request

Sample API request saved to: C:\Users\Nasteho Abdi\employee-attrition-prediction\api\sample_request.json


{'Age': 41,
 'BusinessTravel': 'Travel_Rarely',
 'DailyRate': 1102,
 'Department': 'Sales',
 'DistanceFromHome': 1,
 'Education': 2,
 'EducationField': 'Life Sciences',
 'EnvironmentSatisfaction': 2,
 'Gender': 'Female',
 'HourlyRate': 94,
 'JobInvolvement': 3,
 'JobLevel': 2,
 'JobRole': 'Sales Executive',
 'JobSatisfaction': 4,
 'MaritalStatus': 'Single',
 'MonthlyIncome': 5993,
 'MonthlyRate': 19479,
 'NumCompaniesWorked': 8,
 'OverTime': 'Yes',
 'PercentSalaryHike': 11,
 'PerformanceRating': 3,
 'RelationshipSatisfaction': 1,
 'StockOptionLevel': 0,
 'TotalWorkingYears': 8,
 'TrainingTimesLastYear': 0,
 'WorkLifeBalance': 1,
 'YearsAtCompany': 6,
 'YearsInCurrentRole': 4,
 'YearsSinceLastPromotion': 0,
 'YearsWithCurrManager': 5}

## 11. Build Final Model Metadata

In [13]:
best_model_info = {}

if BEST_MODEL_INFO_PATH.exists():
    with BEST_MODEL_INFO_PATH.open(
        "r",
        encoding="utf-8",
    ) as file:
        best_model_info = json.load(file)

model_metadata = {
    "project": "Employee Attrition Prediction",
    "task": "Binary Classification",
    "target_column": TARGET_COLUMN,
    "negative_class": {
        "value": 0,
        "label": "Stay",
    },
    "positive_class": {
        "value": 1,
        "label": "Leave",
    },
    "model_file": BEST_MODEL_PATH.name,
    "model_type": type(best_model).__name__,
    "feature_count": int(X.shape[1]),
    "input_features": X.columns.tolist(),
    "probability_supported": bool(
        hasattr(best_model, "predict_proba")
    ),
    "risk_thresholds": {
        "low": "probability < 0.30",
        "medium": "0.30 <= probability < 0.60",
        "high": "probability >= 0.60",
    },
    "python_version": platform.python_version(),
    "scikit_learn_version": sklearn.__version__,
    "best_model_evaluation": best_model_info,
}

with MODEL_METADATA_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        model_metadata,
        file,
        indent=4,
    )

print(
    "Model metadata saved to:",
    MODEL_METADATA_PATH.resolve(),
)

Model metadata saved to: C:\Users\Nasteho Abdi\employee-attrition-prediction\models\model_metadata.json


## 12. Test Input Validation Logic

This function checks whether an API request contains all required model features.

In [14]:
EXPECTED_FEATURES = X.columns.tolist()


def validate_employee_payload(
    payload: dict,
) -> tuple[bool, list[str], list[str]]:
    received_features = set(payload.keys())
    expected_features = set(EXPECTED_FEATURES)

    missing = sorted(
        expected_features - received_features
    )

    unexpected = sorted(
        received_features - expected_features
    )

    is_valid = (
        len(missing) == 0
        and len(unexpected) == 0
    )

    return is_valid, missing, unexpected

In [15]:
is_valid, missing, unexpected = (
    validate_employee_payload(
        sample_request
    )
)

print("Payload valid:", is_valid)
print("Missing fields:", missing)
print("Unexpected fields:", unexpected)

Payload valid: True
Missing fields: []
Unexpected fields: []


## 13. Test a Reusable Prediction Function

In [16]:
def predict_employee_attrition(
    payload: dict,
) -> dict:
    is_valid, missing, unexpected = (
        validate_employee_payload(payload)
    )

    if not is_valid:
        raise ValueError({
            "message": "Invalid employee payload.",
            "missing_fields": missing,
            "unexpected_fields": unexpected,
        })

    employee_df = pd.DataFrame(
        [payload],
        columns=EXPECTED_FEATURES,
    )

    prediction = int(
        best_model.predict(employee_df)[0]
    )

    probability = None

    if hasattr(best_model, "predict_proba"):
        probability = float(
            best_model.predict_proba(
                employee_df
            )[0, 1]
        )

    return {
        "prediction": prediction,
        "prediction_label": (
            "Likely to Leave"
            if prediction == 1
            else "Likely to Stay"
        ),
        "attrition_probability": (
            round(probability, 4)
            if probability is not None
            else None
        ),
        "risk_level": (
            probability_to_risk_level(
                probability
            )
        ),
    }


test_result = predict_employee_attrition(
    sample_request
)

test_result

{'prediction': 1,
 'prediction_label': 'Likely to Leave',
 'attrition_probability': 0.7244,
 'risk_level': 'High'}

## 14. Reload the Model Independently

This simulates what FastAPI will do when the server starts.

In [17]:
reloaded_model = joblib.load(
    BEST_MODEL_PATH
)

reloaded_prediction = int(
    reloaded_model.predict(
        pd.DataFrame(
            [sample_request],
            columns=EXPECTED_FEATURES,
        )
    )[0]
)

assert (
    reloaded_prediction
    == test_result["prediction"]
)

print("Independent model reload test passed.")
print("Prediction:", reloaded_prediction)

Independent model reload test passed.
Prediction: 1


# Phase 7 Summary

The final machine-learning model has been packaged and tested for deployment.

## Completed Steps

- Loaded and verified `best_model.joblib`
- Confirmed preprocessing and estimator pipeline steps
- Tested batch predictions
- Tested a single raw employee prediction
- Added probability-based risk levels
- Generated the complete API input schema
- Generated a sample JSON request
- Saved model metadata
- Added payload validation
- Created and tested a reusable prediction function
- Verified independent model reloading

## Generated Files

```text
models/
├── best_model.joblib
├── input_schema.json
└── model_metadata.json

api/
└── sample_request.json
```


